In [1]:
# MTN Management System

In [1]:
import os
import pymysql
import random
import string

In [2]:
PASSWORD = os.environ.get("MYSQL_PASSWORD")

In [3]:
db = os.environ.get("MYSQLPASSWORD")

In [4]:
db = pymysql.connect(
  host= "localhost",
  port= 3306,
  user= "root",
  password= PASSWORD
)
cursor = db.cursor()
if cursor:
  print("Connection Succesful, You may now begin......")

Connection Succesful, You may now begin......


In [5]:
cursor.execute("show databases")
for i in cursor:
  print(i)

('information_schema',)
('mtn',)
('mysql',)
('performance_schema',)
('pop_sys',)
('sakila',)
('shoe_collections',)
('student_data',)
('sys',)
('world',)


In [6]:
cursor.execute("use mtn")

0

In [ ]:
# Customer table - DONE
try:
  cursor.execute("""
  CREATE TABLE Customers(
  cus_id INT PRIMARY KEY NOT NULL,
  name VARCHAR(60) NOT NULL,
  phone_num VARCHAR(15) UNIQUE NOT NULL,
  email VARCHAR(60) NOT NULL,
  address VARCHAR(100) NOT NULL,
  gender ENUM('Male', 'Female') NOT NULL,
  NIN VARCHAR(20) UNIQUE NOT NULL)""")
  if cursor:
    print("Table created succesfully...")
except Exception as e:
  print("Error Message...")

Table created succesfully...


In [10]:
# Plans Table 
try:
  cursor.execute("""
  CREATE TABLE plans(
  plan_id INT AUTO_INCREMENT PRIMARY KEY,
  plan VARCHAR(30) NOT NULL,
  description TEXT NOT NULL,
  price DECIMAL(10, 2) NOT NULL,
  duration INT NOT NULL)""")
  if cursor:
    print("Table Created Successfully....")
except Exception as e:
  print(f"Error Message, the error is {e}")

Table Created Successfully....


In [11]:
# Sub and Bill Table
try:
  cursor.execute("""
  CREATE TABLE Sub(
  sub_id INT PRIMARY KEY,
  cus_id INT NOT NULL,
  plan_id INT NOT NULL,
  start_d DATE,
  end_d DATE,
  amount DECIMAL(10,2) NOT NULL,
  due DATE,
  status ENUM('Pending','Paid','Unpaid') DEFAULT 'Pending',
  FOREIGN KEY (cus_id) REFERENCES Customers(cus_id),
  FOREIGN KEY (plan_id) REFERENCES Plans(plan_id))""")
  if cursor:
    print("Table Created Successfully....")
except Exception as e:
  print(f"Error Message, the error is {e}")

Table Created Successfully....


In [13]:
# Payment Table 
try:
  cursor.execute("""
  CREATE TABLE payment(
  pay_id INT AUTO_INCREMENT PRIMARY KEY,
  sub_id INT NOT NULL,
  pay_date DATE,
  amount DECIMAL(10,2) NOT NULL,
  method ENUM('Cash','Card','Mobile Money','Bank Transfer'),
  FOREIGN KEY (sub_id) REFERENCES Sub(sub_id))""")
  if cursor:
    print("Table Created Successfully....")
except Exception as e:
  print(f"Error Message, the error is {e}")

Table Created Successfully....


In [7]:
cursor.execute("show tables from MTN")
for i in cursor:
  print(i)

('customers',)
('payment',)
('plans',)
('sub',)


In [8]:
sample = string.digits
cus_id ="".join(random.choice(sample) for _ in range(6))
sub_id ="".join(random.choice(sample) for _ in range(5))
pay_id = "".join(random.choice(sample) for _ in range(6))

In [ ]:
def register():
  name = input("Enter full name: ")
  phone_num = input("Enter your phone number: ")
  email = input("Enter your e-mail address: ")
  address = input("Enter your home address: ")
  gender = input("What gender are you(Male/Female): ")
  NIN = input("Enter your National Identification Number: ")
  print(f"""Your name is {name}\nYour phone number is {phone_num}, 
  Your email address is {email}\n Your home address is {address}\n
  You are a {gender}\n and your National Identification Number is {NIN}""")
  conf = input("Are this details correct(Y/N): ")
  if conf.upper() == "Y":
    sample = string.digits
    cus_id ="".join(random.choice(sample) for _ in range(6))
    query = """INSERT INTO Customers(cus_id, name, phone_num, email, address, gender, NIN)
    values(%s, %s, %s, %s, %s, %s, %s) """
    cursor.execute(query, (cus_id, name, phone_num, email, address, gender, NIN))
    db.commit()
    print(f"{name} you have succesfully registered, This is your id:{cus_id}\nWelcome to MTN Smartlife.")
  elif conf == "N":
    print("Okay, please register again and make sure you fill in the correct details")
  else:
    print("Invalid selection.")

def update():
  cus_id = input("Enter the customer id you want to update: ")
  name = input("Enter your Full Name: ")
  phone_num = input("Enter your phone number: ")
  email = input("Enter your e-mail address: ")
  address = input("Enter your home address: ")
  gender = input("What gender are you(Male/Female): ")
  NIN = input("Enter your National Identification Number: ")
  query =""" UPDATE Customers 
  SET name = %s, phone_num = %s, email = %s, address = %s, gender = %s, 
  NIN = %s where cus_id = %s"""
  cursor.execute(query, (name, phone_num, email, address, gender, NIN, cus_id))
  db.commit()
  print("Records have been updated")

def search():
  cus_id = input("Enter your id: ")
  query = " SELECT * FROM Customers WHERE cus_id = %s"
  cursor.execute(query, (cus_id,))
  result = cursor.fetchall()
  if len(result) == 0:
    print("Records not Found")
  else:
    for i in result:
      print(i)

def delete():
  cus_id = input("Enter the customer id you want to delete: ")
  conf = input(f"Are you sure you want to delete customer {cus_id}? (Y/N): ")
  if conf.upper() == "Y":
    query = "DELETE FROM Customers WHERE cus_id = %s"
    cursor.execute(query, (cus_id,))
    db.commit()
    print("Customer record deleted successfully.")
  elif conf.upper() == "N":
    print("Delete cancelled.")
  else:
    print("Invalid Selection!. Delete Cancelled.")

def add_p():
  plan = input("Enter the name of the plan: ")
  description = input("Enter plan info: ")
  price = input("Enter the price: ")
  duration = input("Enter the duration in days: ")
  query = """INSERT INTO PLANS(plan, description, price, duration) 
  VALUES(%s, %s, %s, %s)"""
  cursor.execute(query, (plan, description, price, duration))
  db.commit()
  print("Plan successfully added and is now available for sale.")

def view_p():
  cursor.execute("SELECT * FROM Plans")
  plans = cursor.fetchall()
  for i in plans:
    print(i)

def subscribe():
  cus_id = input("Enter your customer ID: ")
  plan_id = input("Enter plan ID: ")
  cursor.execute("SELECT price, duration FROM Plans WHERE plan_id = %s", (plan_id,))
  plan = cursor.fetchone()
  if not plan:
    print("Invalid plan ID.")
    return
  amount, duration = plan
  sample = string.digits
  sub_id = "".join(random.choice(sample) for _ in range (6))
  query = """INSERT INTO Sub(sub_id, cus_id, plan_id, start_d, end_d, amount, due, status)
  VALUES (%s, %s, %s, CURDATE(), DATE_ADD(CURDATE(), INTERVAL %s DAY), %s, DATE_ADD(CURDATE(), INTERVAL 7 DAY), 'Pending')"""
  cursor.execute(query, (sub_id, cus_id, plan_id, duration, amount))
  db.commit()
  print(f"Subscription created. Your code is {sub_id}. Use this to pay your bill.")

def view_sub():
  cus_id = input("Enter your id: ")
  query = "SELECT * FROM Sub WHERE cus_id = %s"
  cursor.execute(query, (cus_id,))
  result = cursor.fetchall()
  if len(result) == 0:
    print("No subscription found")
  else:
    for i in result:
      print(i)

def pay():
  sub_id = input("Enter your subscription code: ")
  cursor.execute("SELECT amount FROM Sub WHERE sub_id = %s AND status = 'Pending'", (sub_id,))
  sub = cursor.fetchone()
  if not sub:
    print("Invalid subscription code or subscription already paid.")
    return
  amount = sub[0]
  method = input("Enter payment method (Cash/Card/Mobile Money/Bank Transfer): ")
  query = """INSERT INTO Payment(sub_id, pay_date, amount, method)
  VALUES (%s, CURDATE(), %s, %s)"""
  cursor.execute(query, (sub_id, amount, method))
  cursor.execute("UPDATE Sub SET status = 'Paid' WHERE sub_id = %s", (sub_id,))
  db.commit()
  print("Payment successful. Subscription activated.")

def view_pay():
  cus_id = input("Enter your customer ID: ")
  query = """SELECT Payment.* FROM Payment JOIN Sub ON Payment.sub_id = Sub.sub_id 
  WHERE Sub.cus_id = %s"""
  cursor.execute(query, (cus_id,))
  result = cursor.fetchall()
  if len(result) == 0:
    print("No payments found for this customer.")
  else:
    for pay in result:
      print(pay)

def main():
  while True:
    print("##############################################")
    print("#              MTN SMARTLIFE                 #")
    print("##############################################")
    print("\n1. << Register\n 2. << Customer Sign in\n 3. << Admin Sign in\n 4.Exit.")
    beg = input("Enter an option: ")
    if beg == "1":
      register()
    elif beg == "2":
      cus_id = input("Please enter your customer id: ")
      cursor.execute("SELECT name from Customers where cus_id = %s", (cus_id,))
      stmg = cursor.fetchone()
      print(f"Welcome to MTN Smartlife {stmg[0]}. What service can we offer you today: ")
      print("""\n1.<< Update your Information.\n2.<< View available Plans.\n3.<< Suscribe for a plan.\n
4.View all Subscriptions.\n5.<< Make a Payment.\n6.<< View Payment Hstory\n7.<< Exit """)
      ans = input("Enter an option: ")
      if ans == "1":
        update()
      elif ans == "2":
        view_p()
      elif ans == "3":
        subscribe()
      elif ans == "4":
        view_sub()
      elif ans == "5":
        pay()
      elif ans == "6":
        view_pay()
      elif ans == "7":
        print("Thank you for using our MTN Smartlife services.")
      else:
        print("Invalid Selection...")
        break
    elif beg == "3":
      pss = 1692
      passw = int(input("Enter the password: "))
      if passw == pss:
        print("Welcome to MTN Smartlife Admin, what would you like to do today.")
        print("""\n1.<< Delete a customer id.\n2.<<Add a Plan.\n3.<< Update your Information.\n4.<< View available Plans.\n5.<< Suscribe for a plan.\n6.View all Subscriptions.\n7.<< Make a Payment.\n8.<< View Payment Hstory\n9.<< Exit""")
        anss = input("Enter an option: ")
        if anss == "1":
          delete()
        elif anss == "2":
          add_p()
        elif anss == "3":
          update()
        elif anss == "4":
          view_p()
        elif anss == "5":
          subscribe()
        elif anss == "6":
          view_sub()
        elif anss == "7":
          pay()
        elif anss == "8":
          view_pay()
        elif anss == "9":
          print("Thank you for using our services...")
          break
      elif passw != pss:
        print("Wrong Password..") 

if __name__ == "__main__":
  main()

##############################################
#              MTN SMARTLIFE                 #
##############################################

1. << Register
 2. << Customer Sign in
 3. << Admin Sign in
 4.Exit.


Enter an option:  2
Please enter your customer id:  634340


Welcome to MTN Smartlife Lawal. What service can we offer you today: 

1.<< Update your Information.
2.<< View available Plans.
3.<< Suscribe for a plan.

4.View all Subscriptions.
5.<< Make a Payment.
6.<< View Payment Hstory
7.<< Exit 


Enter an option:  3
Enter your customer ID:  634340
Enter plan ID:  1


Subscription created. Your code is 182179. Use this to pay your bill.
##############################################
#              MTN SMARTLIFE                 #
##############################################

1. << Register
 2. << Customer Sign in
 3. << Admin Sign in
 4.Exit.


...

Welcome to MTN Smartlife Lawal. What service can we offer you today: 

1.<< Update your Information.
2.<< View available Plans.
3.<< Suscribe for a plan.

4.View all Subscriptions.
5.<< Make a Payment.
6.<< View Payment Hstory
7.<< Exit 


Enter an option:  5
Enter your subscription code:  182179
Enter payment method (Cash/Card/Mobile Money/Bank Transfer):  Cash


Payment successful. Subscription activated.
##############################################
#              MTN SMARTLIFE                 #
##############################################

1. << Register
 2. << Customer Sign in
 3. << Admin Sign in
 4.Exit.


Enter an option:  4


##############################################
#              MTN SMARTLIFE                 #
##############################################

1. << Register
 2. << Customer Sign in
 3. << Admin Sign in
 4.Exit.
